In [2]:
import re
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

raw = pd.read_csv("used-car-price-predictor/data/raw/Car details v3.csv")
df = raw.copy()
print("Start:", df.shape)

Start: (8128, 13)


In [3]:
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} duplicate rows -> {len(df)} left")

Removed 1202 duplicate rows -> 6926 left


In [4]:
num = r"(\d+\.?\d*)"   # first number found in the text

df["mileage_kmpl"]  = pd.to_numeric(df["mileage"].str.extract(num)[0], errors="coerce")
df["engine_cc"]     = pd.to_numeric(df["engine"].str.extract(num)[0], errors="coerce")
df["max_power_bhp"] = pd.to_numeric(df["max_power"].str.extract(num)[0], errors="coerce")

def torque_to_nm(t):
    if pd.isna(t):
        return np.nan
    t = str(t).lower().replace(",", "")
    m = re.search(r"(\d+\.?\d*)", t)
    if not m:
        return np.nan
    value = float(m.group(1))
    if "kgm" in t:
        value *= 9.80665          # kgm -> Nm
    return value

df["torque_nm"] = df["torque"].apply(torque_to_nm)
df.loc[~df["torque_nm"].between(30, 1500), "torque_nm"] = np.nan   # impossible values

df = df.drop(columns=["mileage", "engine", "max_power", "torque"])
df[["mileage_kmpl", "engine_cc", "max_power_bhp", "torque_nm"]].describe().T

,count,mean,std,min,25%,50%,75%,max
mileage_kmpl,6718.0,19.465310,4.049150,0.00000,16.8,19.44,22.50000,42.00000
engine_cc,6718.0,1430.891337,493.493277,624.00000,1197.0,1248.00,1498.00000,3604.00000
max_power_bhp,6720.0,87.726919,31.771619,0.00000,67.1,81.83,100.00000,400.00000
torque_nm,6715.0,174.402545,104.856557,47.07192,110.0,160.00,200.05566,1421.96425


In [5]:
df["brand"] = df["name"].str.split().str[0]
df["brand"] = df["brand"].replace({"Land": "Land Rover"})
print(df["brand"].value_counts())

print(df.groupby("fuel")["mileage_kmpl"].mean().round(1))   # CNG/LPG are in km/kg, not kmpl

brand
Maruti           2165
Hyundai          1267
Mahindra          723
Tata              647
Honda             362
Ford              361
Toyota            357
Chevrolet         216
Renault           206
Volkswagen        174
Nissan             73
Skoda              70
Datsun             57
BMW                47
Mercedes-Benz      46
Fiat               44
Audi               33
Jeep               22
Mitsubishi         11
Volvo               9
Jaguar              8
Isuzu               4
Ambassador          4
Force               4
Kia                 3
Land Rover          3
Daewoo              3
MG                  3
Ashok               1
Lexus               1
Opel                1
Peugeot             1
Name: count, dtype: int64
fuel
CNG       24.0
Diesel    19.7
LPG       18.6
Petrol    19.1
Name: mileage_kmpl, dtype: float64


In [6]:
print(df["owner"].value_counts())
print("\nkm_driven > 500,000:", (df["km_driven"] > 500_000).sum())
print("year < 1995:", (df["year"] < 1995).sum())

df.sort_values("selling_price", ascending=False).head(10)[["name", "year", "selling_price", "km_driven"]]

owner
First Owner             4242
Second Owner            1974
Third Owner              536
Fourth & Above Owner     169
Test Drive Car             5
Name: count, dtype: int64

km_driven > 500,000: 3
year < 1995: 5


,name,year,selling_price,km_driven
170,Volvo XC90 T8 Excellence BSIV,2017,10000000,30000
2689,BMW X7 xDrive 30d DPE,2020,7200000,5000
4288,Audi A6 35 TFSI Matrix,2019,6523000,23600
4286,Audi A6 35 TFSI Matrix,2019,6223000,7800
136,Mercedes-Benz S-Class S 350 CDI,2017,6000000,37000
4143,BMW 6 Series GT 630d Luxury Line,2018,6000000,27000
1017,BMW 6 Series GT 630d Luxury Line,2018,6000000,28156
4287,Audi A6 35 TFSI Matrix,2019,5923000,11500
148,Mercedes-Benz S-Class S 350 CDI,2017,5850000,47000
5384,BMW 6 Series GT 630d Luxury Line,2018,5830000,30000


In [7]:
before = len(df)

# 1. "Test Drive Car" is not a normal ownership type and has very few rows
df = df[df["owner"] != "Test Drive Car"]

# 2. Unrealistic usage (data entry errors)
df = df[df["km_driven"] <= 500_000]

# 3. Zero or negative specs are invalid, so mark them missing
for col in ["mileage_kmpl", "engine_cc", "max_power_bhp"]:
    df.loc[df[col] <= 0, col] = np.nan

df = df.reset_index(drop=True)
print(f"Removed {before - len(df)} rows -> {len(df)} left")

Removed 8 rows -> 6918 left


In [8]:
print(df.isnull().sum())

assert df["selling_price"].min() > 0
assert df["year"].between(1980, 2022).all()
assert df["km_driven"].between(0, 500_000).all()
assert df["seats"].dropna().between(2, 14).all()
print("Validation passed")

name               0
year               0
selling_price      0
km_driven          0
fuel               0
seller_type        0
transmission       0
owner              0
seats            208
mileage_kmpl     223
engine_cc        208
max_power_bhp    209
torque_nm        211
brand              0
dtype: int64
Validation passed


In [10]:
df.to_csv("used-car-price-predictor/data/raw/Car details v3.csv", index=False)
print("Saved:", df.shape)
df.info()

Saved: (6918, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6918 entries, 0 to 6917
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   name           6918 non-null   object 
 1   year           6918 non-null   int64  
 2   selling_price  6918 non-null   int64  
 3   km_driven      6918 non-null   int64  
 4   fuel           6918 non-null   object 
 5   seller_type    6918 non-null   object 
 6   transmission   6918 non-null   object 
 7   owner          6918 non-null   object 
 8   seats          6710 non-null   float64
 9   mileage_kmpl   6695 non-null   float64
 10  engine_cc      6710 non-null   float64
 11  max_power_bhp  6709 non-null   float64
 12  torque_nm      6707 non-null   float64
 13  brand          6918 non-null   object 
dtypes: float64(5), int64(3), object(6)
memory usage: 756.8+ KB


In [12]:
import os

os.makedirs("used-car-price-predictor/data/processed", exist_ok=True)

df.to_csv("used-car-price-predictor/data/processed/cars_clean.csv", index=False)

print("Cleaned dataset saved successfully!")
print("Shape:", df.shape)

Cleaned dataset saved successfully!
Shape: (6918, 14)
